## Import library

In [12]:
import os
import warnings
import json
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from pathlib import Path

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    f1_score
)

import mlflow
from mlflow import MlflowClient

warnings.filterwarnings("ignore")

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

RANDOM_STATE = 42

# Env Path
ENV_PATH = Path.cwd().parents[0] / "paylater_engineering" / ".env"
load_dotenv(ENV_PATH)

print("AI Engineering Research Environment Initialized")

AI Engineering Research Environment Initialized


## Database & MLFlow configuration

In [13]:
DATABASE_URL = os.getenv("DATABASE_URL")

DB_SCHEMA = os.getenv("POSTGRES_SCHEMA")
DB_TABLE = os.getenv("POSTGRES_TABLE")

MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Setup mlflow client
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

# Setup postgres connection
engine = create_engine(DATABASE_URL, pool_pre_ping=True)

print(f"Database: DB_SCHEMA={DB_SCHEMA}, DB_TABLE={DB_TABLE}")
print(f"MLFlow: {mlflow.get_tracking_uri()}")

Database: DB_SCHEMA=paylater, DB_TABLE=database
MLFlow: http://localhost:5000


## Research Overview & Business Objectives

In [14]:
research_framework = pd.DataFrame([
    {
        "Research_ID": "R01",
        "Research_Name": "OJK TWP90 Default Early Warning",
        "ML_Task": "Classification",
        "Target": "twp90_default_status",
        "Business_Objective": "Identify borrowers approaching 90 DPD"
    },
    {
        "Research_ID": "R02",
        "Research_Name": "Gen Z DTI & Financial Distress",
        "ML_Task": "Multi-output Regression",
        "Target": "debt_to_income_ratio, financial_distress_index",
        "Business_Objective": "Forecast borrower financial vulnerability"
    },
    {
        "Research_ID": "R03",
        "Research_Name": "Structural Insolvency Root Cause",
        "ML_Task": "Multiclass Classification",
        "Target": "primary_default_root_cause",
        "Business_Objective": "Identify structural financial distress causes"
    }
])

display(research_framework)

,Research_ID,Research_Name,ML_Task,Target,Business_Objective
0,R01,OJK TWP90 Default Early Warning,Classification,twp90_default_status,Identify borrowers approaching 90 DPD
1,R02,Gen Z DTI & Financial Distress,Multi-output Regression,"debt_to_income_ratio, financial_distress_index",Forecast borrower financial vulnerability
2,R03,Structural Insolvency Root Cause,Multiclass Classification,primary_default_root_cause,Identify structural financial distress causes


## Business research hypotheses

In [15]:
research_hypotheses = {
    "H1": "Higher DTI is associated with increased default risk.",
    "H2": "Higher DSR is associated with financial distress.",
    "H3": "Multiple active lending applications are associated "
          "with higher borrower financial vulnerability.",
    "H4": "Negative disposable income is associated with "
          "higher delinquency risk.",
    "H5": "Higher credit utilization is associated with "
          "increased repayment vulnerability.",
    "H6": "Borrower financial characteristics differ "
          "significantly across structural root causes."
}

for hypothesis, statement in research_hypotheses.items():
    print(f"{hypothesis}: {statement}")

H1: Higher DTI is associated with increased default risk.
H2: Higher DSR is associated with financial distress.
H3: Multiple active lending applications are associated with higher borrower financial vulnerability.
H4: Negative disposable income is associated with higher delinquency risk.
H5: Higher credit utilization is associated with increased repayment vulnerability.
H6: Borrower financial characteristics differ significantly across structural root causes.


## Dataset Loading & Data Validation

In [16]:
query = f"""
SELECT *
FROM "{DB_SCHEMA}"."{DB_TABLE}"
"""

df = pd.read_sql(
    query,
    engine
)

print("Dataset successfully loaded")
print("Shape:", df.shape)

display(df.head())

Dataset successfully loaded
Shape: (10000, 32)


,borrower_id,age,gender,province,education_level,employment_status,monthly_income_idr,regional_minimum_wage_umr_idr,num_active_lending_apps,total_loan_outstanding_idr,credit_limit_granted_idr,loan_tenor_months,daily_interest_rate_pct,primary_loan_purpose,debt_to_income_ratio,income_to_umr_ratio,credit_limit_utilization_ratio,monthly_installment_estimate_idr,debt_service_ratio,disposable_income_after_debt_idr,days_past_due,financial_distress_index,twp90_default_status,primary_default_root_cause,cot_financial_psychology_rationale,confidence_score,risk_score,risk_short,is_high_risk,distress_bin,created_at,updated_at
0,ID_DEB_00264B68D1,24,Female,Bali,High School / SMK,Micro-Entrepreneur / UMKM,"6,420,000.0000","2,950,000.0000",2,"8,380,000.0000","12,280,000.0000",1,0.3500,Daily Living & Emergency Buffer,1.3100,2.1800,0.6824,"9,259,900.0000",1.4424,"-2,839,900.0000",67,3.9700,Doubtful-DPD61-90,High-Interest-Rate-Compounding,"Debitur 'ID_DEB_00264B68D1' (Female, usia 24 t...",0.9610,3,Doubt(61-90),True,"(3.686, 4.588]",2026-11-08 13:57:53.663737,2026-11-08 13:57:53.663737
1,ID_DEB_0DE3583B12,28,Female,Bali,Bachelor (S1),Formal Private Employee,"8,630,000.0000","2,950,000.0000",1,"20,110,000.0000","29,060,000.0000",6,0.3000,Daily Living & Emergency Buffer,2.3300,2.9300,0.6920,"5,161,567.0000",0.5981,"3,468,433.0000",11,4.8700,Special-Mention-DPD1-30,High-Interest-Rate-Compounding,"Debitur 'ID_DEB_0DE3583B12' (Female, usia 28 t...",0.9750,1,SM(1-30),False,"(4.588, 5.49]",2026-11-08 13:57:53.663737,2026-11-08 13:57:53.663737
2,ID_DEB_0F661FADC0,20,Female,Bali,High School / SMK,Civil Servant / BUMN,"5,720,000.0000","2,950,000.0000",4,"14,960,000.0000","23,900,000.0000",1,0.3200,Daily Living & Emergency Buffer,2.6200,1.9400,0.6259,"16,396,160.0000",2.8665,"-10,676,160.0000",180,6.0700,Default-Macet-TWP90,High-Interest-Rate-Compounding,"Debitur 'ID_DEB_0F661FADC0' (Female, usia 20 t...",0.9900,4,Default Rate(>90),True,"(5.49, 6.392]",2026-11-08 13:57:53.663737,2026-11-08 13:57:53.663737
3,ID_DEB_10C74D3F21,31,Female,Bali,Bachelor (S1),Civil Servant / BUMN,"11,780,000.0000","2,950,000.0000",3,"7,620,000.0000","26,130,000.0000",1,0.2600,Daily Living & Emergency Buffer,0.6500,3.9900,0.2916,"8,214,360.0000",0.6973,"3,565,640.0000",24,2.3500,Special-Mention-DPD1-30,Compulsive-FOMO-Spending,"Debitur 'ID_DEB_10C74D3F21' (Female, usia 31 t...",0.9770,1,SM(1-30),False,"(1.882, 2.784]",2026-11-08 13:57:53.663737,2026-11-08 13:57:53.663737
4,ID_DEB_12CFCAE01F,22,Male,Bali,Bachelor (S1),Formal Private Employee,"6,490,000.0000","2,950,000.0000",2,"11,080,000.0000","16,420,000.0000",9,0.2300,Daily Living & Emergency Buffer,1.7100,2.2000,0.6748,"1,995,631.0000",0.3075,"4,494,369.0000",0,4.4300,Current-Performing,Compulsive-FOMO-Spending,"Debitur 'ID_DEB_12CFCAE01F' (Male, usia 22 tah...",0.9800,0,Current,False,"(3.686, 4.588]",2026-11-08 13:57:53.663737,2026-11-08 13:57:53.663737


In [21]:
print(f"Missing values: {df.isnull().sum().sum()}")

Missing values: 0


## Data quality assessment

In [22]:
quality_report = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Features",
        "Duplicate Rows",
        "Unique Borrowers",
        "Missing Cells",
        "Missing Percentage"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df["borrower_id"].nunique(),
        df.isnull().sum().sum(),
        df.isnull().sum().sum() / df.size * 100
    ]
})

display(quality_report)

,Metric,Value
0,Total Records,"10,000.0000"
1,Total Features,32.0000
2,Duplicate Rows,0.0000
3,Unique Borrowers,"10,000.0000"
4,Missing Cells,0.0000
5,Missing Percentage,0.0000


In [23]:
# Validate financial variables
financial_columns = [
    "monthly_income_idr",
    "total_loan_outstanding_idr",
    "credit_limit_granted_idr",
    "debt_to_income_ratio",
    "income_to_umr_ratio",
    "credit_limit_utilization_ratio",
    "monthly_installment_estimate_idr",
    "debt_service_ratio",
    "disposable_income_after_debt_idr",
    "financial_distress_index",
    "days_past_due",
    "risk_score"
]

financial_summary = df[
    financial_columns
].describe().T

display(financial_summary)

,count,mean,std,min,25%,50%,75%,max
monthly_income_idr,"10,000.0000","6,251,306.0000","3,813,988.3751","1,500,000.0000","3,490,000.0000","5,280,000.0000","7,750,000.0000","28,000,000.0000"
total_loan_outstanding_idr,"10,000.0000","9,901,590.0000","8,605,004.3999","510,000.0000","4,237,500.0000","7,260,000.0000","12,582,500.0000","65,000,000.0000"
credit_limit_granted_idr,"10,000.0000","16,695,954.0000","11,960,197.2075","1,890,000.0000","8,447,500.0000","13,300,000.0000","21,160,000.0000","80,000,000.0000"
debt_to_income_ratio,"10,000.0000",1.5888,0.8630,0.2500,0.9200,1.4000,2.0625,5.0700
income_to_umr_ratio,"10,000.0000",1.9452,0.8417,0.6000,1.2600,1.8000,2.5400,6.1500
credit_limit_utilization_ratio,"10,000.0000",0.5935,0.2442,0.1995,0.4089,0.5619,0.7645,1.2492
monthly_installment_estimate_idr,"10,000.0000","5,090,861.1426","6,534,838.5062","85,307.0000","1,357,372.5000","2,823,700.0000","6,127,509.7500","71,630,000.0000"
debt_service_ratio,"10,000.0000",0.8164,0.7955,0.0327,0.2827,0.5334,1.0515,5.3506
disposable_income_after_debt_idr,"10,000.0000","1,160,444.8574","5,816,387.9280","-54,479,080.0000","-237,216.7500","1,916,308.5000","3,929,806.5000","24,600,412.0000"
financial_distress_index,"10,000.0000",4.7285,2.0612,0.9800,3.1800,4.2800,5.9700,10.0000


In [25]:
# Check for impossible or suspicious observations.
validation_checks = {
    "Negative Monthly Income":
        (df["monthly_income_idr"] < 0).sum(),

    "Negative Outstanding Debt":
        (df["total_loan_outstanding_idr"] < 0).sum(),

    "Negative Credit Limit":
        (df["credit_limit_granted_idr"] < 0).sum(),

    "Negative DPD":
        (df["days_past_due"] < 0).sum(),

    "Invalid Age":
        (~df["age"].between(18, 100)).sum(),

    "Distress Outside 0-10":
        (~df["financial_distress_index"].between(0, 10)).sum()
}

display(pd.DataFrame(validation_checks.items(), columns=["Validation", "Invalid Count"]))

,Validation,Invalid Count
0,Negative Monthly Income,0
1,Negative Outstanding Debt,0
2,Negative Credit Limit,0
3,Negative DPD,0
4,Invalid Age,0
5,Distress Outside 0-10,0


## Section 03 - Feature Dictionary & Target Definitions

In [26]:
feature_dictionary = {
    "Demographic": [
        "age",
        "gender",
        "province",
        "education_level",
        "employment_status"
    ],

    "Income": [
        "monthly_income_idr",
        "regional_minimum_wage_umr_idr",
        "income_to_umr_ratio"
    ],

    "Credit Exposure": [
        "num_active_lending_apps",
        "total_loan_outstanding_idr",
        "credit_limit_granted_idr",
        "credit_limit_utilization_ratio",
        "loan_tenor_months"
    ],

    "Repayment Capacity": [
        "monthly_installment_estimate_idr",
        "debt_service_ratio",
        "disposable_income_after_debt_idr"
    ],

    "Financial Risk": [
        "debt_to_income_ratio",
        "financial_distress_index",
        "days_past_due"
    ],

    "Targets": [
        "twp90_default_status",
        "primary_default_root_cause",
        "debt_to_income_ratio",
        "financial_distress_index"
    ]
}

for category, columns in feature_dictionary.items():
    available = [col for col in columns if col in df.columns]
    print(f"\n{category}")
    print("-" * 50)

    for column in available:
        print(column)


Demographic
--------------------------------------------------
age
gender
province
education_level
employment_status

Income
--------------------------------------------------
monthly_income_idr
regional_minimum_wage_umr_idr
income_to_umr_ratio

Credit Exposure
--------------------------------------------------
num_active_lending_apps
total_loan_outstanding_idr
credit_limit_granted_idr
credit_limit_utilization_ratio
loan_tenor_months

Repayment Capacity
--------------------------------------------------
monthly_installment_estimate_idr
debt_service_ratio
disposable_income_after_debt_idr

Financial Risk
--------------------------------------------------
debt_to_income_ratio
financial_distress_index
days_past_due

Targets
--------------------------------------------------
twp90_default_status
primary_default_root_cause
debt_to_income_ratio
financial_distress_index


## Target distribution inspection

In [27]:
target_columns = [
    "twp90_default_status",
    "primary_default_root_cause"
]

for target in target_columns:

    print(f"\nTarget: {target}")
    print("=" * 60)

    display(
        df[target]
        .value_counts(dropna=False)
        .to_frame("Count")
        .assign(
            Percentage=lambda x:
                x["Count"] / len(df) * 100
        )
    )


Target: twp90_default_status


,Count,Percentage
twp90_default_status,,
Current-Performing,3508,35.0800
Default-Macet-TWP90,2538,25.3800
Special-Mention-DPD1-30,1731,17.3100
Substandard-DPD31-60,1367,13.6700
Doubtful-DPD61-90,856,8.5600



Target: primary_default_root_cause


,Count,Percentage
primary_default_root_cause,,
Compulsive-FOMO-Spending,3588,35.8800
Gali-Lubang-Debt-Spiral,2250,22.5000
Income-Shock-Underemployment,1989,19.8900
High-Interest-Rate-Compounding,1616,16.1600
Speculative-Losses,557,5.5700
